In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

import json

In [ ]:
df_train = pd.read_parquet("03_datasets/binned_train.parquet")
df_test = pd.read_parquet("03_datasets/binned_test.parquet")

In [ ]:
df_train[["Opening", "ECO"]].head(10)

## Mean Encoding

In [ ]:
df_train["OpeningType"] = df_train["Opening"].str.split(":", expand=True)[0]
df_train["ECOType"] = df_train["ECO"].str[:2]

df_test["OpeningType"] = df_test["Opening"].str.split(":", expand=True)[0]
df_test["ECOType"] = df_test["ECO"].str[:2]

In [ ]:
opening_features = ["Opening", "ECO", "OpeningType", "ECOType"]
for feature in opening_features:
    
    # Распространенные дебюты
    common_openings = (
        df_train[feature]
        .value_counts()
        .where(lambda x: x >= 100).dropna()
        .index
    )
    
    # Считаем средние
    means_dict = (
        df_train
        .groupby(feature)
        .agg({"Elo": "mean"})
        .loc[common_openings]
        .squeeze().to_dict()
    )
    
    # Среднее для редких
    mean_if_rare = df_train["Elo"].where(
        ~df_train["Opening"].isin(common_openings)
    ).mean()
    
    # Применяем
    df_train[feature] = (
        df_train[feature]
        .map(means_dict)
        .fillna(mean_if_rare)
    )
    
    df_test[feature] = (
        df_test[feature]
        .map(means_dict)
        .fillna(mean_if_rare)
    )

In [ ]:
df_train.to_parquet("03_datasets/mean_train.parquet")
df_test.to_parquet("03_datasets/mean_test.parquet")

## Корзина